# RFM Segmentation

In [1]:
import pandas as pd

In [2]:
# read clean csv dataset 
cleaned = pd.read_csv('../data/cleaned/retail_cleaned.csv')
cleaned['InvoiceDate'] = pd.to_datetime(cleaned['InvoiceDate'])

# checking the invoice date, is all the data exist?
print(cleaned.shape)
print(cleaned['InvoiceDate'].min(), '->', cleaned['InvoiceDate'].max())

(779425, 11)
2009-12-01 07:45:00 -> 2011-12-09 12:50:00


In [3]:
# Snapshot date = one day after the last transaction in the dataset
snapshot_date = cleaned['InvoiceDate'].max() + pd.Timedelta(days=1)
print('Snapshot date:', snapshot_date)

# assign the rfm of all customer
rfm = cleaned.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('LineRevenue', 'sum')
).reset_index()


print(rfm.shape)
print(rfm.head(10))
print(rfm.describe())

Snapshot date: 2011-12-10 12:50:00
(5878, 4)
   Customer ID  Recency  Frequency  Monetary
0      12346.0      326         12  77556.46
1      12347.0        2          8   4921.53
2      12348.0       75          5   2019.40
3      12349.0       19          4   4428.69
4      12350.0      310          1    334.40
5      12351.0      375          1    300.93
6      12352.0       36         10   2849.84
7      12353.0      204          2    406.76
8      12354.0      232          1   1079.40
9      12355.0      214          2    947.61
        Customer ID      Recency    Frequency       Monetary
count   5878.000000  5878.000000  5878.000000    5878.000000
mean   15315.313542   201.331916     6.289384    2955.904095
std     1715.572666   209.338707    13.009406   14440.852688
min    12346.000000     1.000000     1.000000       2.950000
25%    13833.250000    26.000000     1.000000     342.280000
50%    15314.500000    96.000000     3.000000     867.740000
75%    16797.750000   380.000000 

In [23]:
# frequency and monetary has an extrem outlier, and this one to check if it is a wholesaler
print('Sort Based on: Monetary\n',rfm.sort_values('Monetary', ascending=False).head(5))
print('\n\nSort Based on: Frequency\n',rfm.sort_values('Frequency', ascending=False).head(5))

Sort Based on: Monetary
       Customer ID  Recency  Frequency   Monetary
5692      18102.0        1        145  580987.04
2277      14646.0        2        151  528602.52
1789      14156.0       10        156  313437.62
2538      14911.0        1        398  291420.81
5050      17450.0        8         51  244784.25


Sort Based on: Frequency
       Customer ID  Recency  Frequency   Monetary
2538      14911.0        1        398  291420.81
400       12748.0        1        336   53539.64
5433      17841.0        2        211   68545.25
2935      15311.0        1        208  114966.42
739       13089.0        3        203  113416.91


In [ ]:
# Score R, F, M into quintiles (1-5), 5,4,3,2,1 in order and 5 is always the best score in rfm

# Recency: lower is bettter, reverse the labels (5 = most recent)
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

# Frequency: use rank(method='first) to break ties, since many customers share the same low frequency, 5 is the most frequency
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])

# Monetary: higher is better, 5 is the most high monetary 
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

print(rfm.head(10))
print(rfm[['R_score', 'F_score', 'M_score']].apply(lambda x: x.value_counts()).T)

   Customer ID  Recency  Frequency  Monetary R_score F_score M_score
0      12346.0      326         12  77556.46       2       5       5
1      12347.0        2          8   4921.53       5       4       5
2      12348.0       75          5   2019.40       3       4       4
3      12349.0       19          4   4428.69       5       3       5
4      12350.0      310          1    334.40       2       1       2
5      12351.0      375          1    300.93       2       1       2
6      12352.0       36         10   2849.84       4       5       4
7      12353.0      204          2    406.76       2       2       2
8      12354.0      232          1   1079.40       2       1       3
9      12355.0      214          2    947.61       2       2       3
            1     2     3     4     5
R_score  1175  1172  1167  1176  1188
F_score  1176  1175  1176  1175  1176
M_score  1176  1175  1176  1175  1176


In [26]:
# Combine into a single RFM score string
rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

# Map into readable segment names
def segment_customer(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)    

print(rfm[['Customer ID', 'R_score', 'F_score', 'M_score', 'RFM_Score', 'Segment']].head(10))
print(rfm['Segment'].value_counts())

   Customer ID R_score F_score M_score RFM_Score          Segment
0      12346.0       2       5       5       255          At Risk
1      12347.0       5       4       5       545        Champions
2      12348.0       3       4       4       344  Loyal Customers
3      12349.0       5       3       5       535  Loyal Customers
4      12350.0       2       1       2       212             Lost
5      12351.0       2       1       2       212             Lost
6      12352.0       4       5       4       454        Champions
7      12353.0       2       2       2       222             Lost
8      12354.0       2       1       3       213             Lost
9      12355.0       2       2       3       223             Lost
Segment
Lost               1523
Loyal Customers    1406
Champions          1297
At Risk             824
New Customers       443
Needs Attention     385
Name: count, dtype: int64


In [27]:
# Revenue contribution by segment
segment_revenue = rfm.groupby('Segment').agg(
    CustomerCount=('Customer ID', 'count'),
    TotalRevenue=('Monetary', 'sum'),
    AvgRevenue=('Monetary', 'mean')
).sort_values('TotalRevenue', ascending=False)

segment_revenue['RevenuePct'] = (segment_revenue['TotalRevenue'] / segment_revenue['TotalRevenue'].sum() * 100).round(2)

print(segment_revenue)

                 CustomerCount  TotalRevenue   AvgRevenue  RevenuePct
Segment                                                              
Champions                 1297  1.185959e+07  9143.863913       68.26
Loyal Customers           1406  2.674785e+06  1902.407536       15.39
At Risk                    824  1.589384e+06  1928.864456        9.15
Lost                      1523  6.544267e+05   429.695799        3.77
New Customers              443  3.922670e+05   885.478600        2.26
Needs Attention            385  2.043497e+05   530.778553        1.18


In [28]:
rfm.to_csv('../data/cleaned/rfm_segments.csv', index=False)

In [29]:
# Repeat Purchase Rate Over Time
first_purchase = cleaned.groupby('Customer ID')['InvoiceDate'].min().reset_index()
first_purchase.columns = ['Customer ID', 'FirstPurchaseDate']

# Merge bact to main data
cleaned_merged = cleaned.merge(first_purchase, on='Customer ID')

# Flag: is this transaction happening in the customer's first month, or later (repeat)?
cleaned_merged['OrderMonth'] = cleaned_merged['InvoiceDate'].dt.to_period('M')
cleaned_merged['FirstPurchaseMonth'] = cleaned_merged['FirstPurchaseDate'].dt.to_period('M')
cleaned_merged['CustomerType'] = cleaned_merged.apply(
    lambda x: 'New' if x['OrderMonth'] == x['FirstPurchaseMonth'] else 'Repeat', axis=1
)

# Unique customers per month, split by New vs Repeat
monthly_customer_type = cleaned_merged.groupby(['OrderMonth', 'CustomerType'])['Customer ID'].nunique().unstack(fill_value=0)
monthly_customer_type['RepeatRate'] = (monthly_customer_type['Repeat'] / (monthly_customer_type['New'] + monthly_customer_type['Repeat']) * 100).round(2)

print(monthly_customer_type)

CustomerType  New  Repeat  RepeatRate
OrderMonth                           
2009-12       955       0        0.00
2010-01       383     337       46.81
2010-02       374     398       51.55
2010-03       443     614       58.09
2010-04       294     648       68.79
2010-05       254     712       73.71
2010-06       270     771       74.06
2010-07       186     742       79.96
2010-08       162     749       82.22
2010-09       243     902       78.78
2010-10       377    1120       74.82
2010-11       325    1282       79.78
2010-12        76     809       91.41
2011-01        71     670       90.42
2011-02       124     634       83.64
2011-03       179     795       81.62
2011-04       106     750       87.62
2011-05       111     945       89.49
2011-06       108     883       89.10
2011-07       102     847       89.25
2011-08       106     829       88.66
2011-09       189    1077       85.07
2011-10       221    1143       83.80
2011-11       191    1473       88.52
2011-12     